In [ ]:
# Fixed file mapping - consistent underscores
files = {
    "MModal_Baseline": "model_v1_predictions.npz",
    "MM_augmentation": "roc_data_augmentation.npz", 
    "MM_attention": "roc_data_augmentation_attention.npz",
    "MM_transformer": "roc_data_transformer.npz",
    "MM_transformer_attention": "roc_data_combined.npz",
    "MM_ViT16": "roc_data_transformer_ViT-16.npz",
}

print("Updated file mapping:")
for model, file in files.items():
    print(f"{model}: {file}")

In [ ]:
import os

# Quick file existence check
for model, fname in files.items():
    if os.path.exists(fname):
        print(f"[OK] {model}")
    else:
        print(f"[MISSING] {model}: {fname}")

In [ ]:
# Fixed color mapping - matching the underscore naming
colors = {
    "MModal_Baseline": "#1f77b4",
    "MM_augmentation": "#ff7f0e", 
    "MM_attention": "#2ca02c",
    "MM_transformer": "#d62728",
    "MM_transformer_attention": "#8c564b",
    "MM_ViT16": "#9467bd",
}

print("Color mapping:")
for model in files.keys():
    print(f"{model}: {colors[model]}")

In [ ]:
import numpy as np
from sklearn.metrics import roc_auc_score

def auc_from_predictions(npz_path):
    """Extract AUC from baseline model (has y_test, y_score)"""
    d = np.load(npz_path, allow_pickle=True)
    y_true = d["y_test"]
    y_score = d["y_score"]
    auc = roc_auc_score(y_true, y_score, multi_class="ovr", average="macro")
    d.close()
    return float(auc)

def auc_from_roc_npz(npz_path):
    """Extract AUC from ROC data files (has roc_auc dict)"""
    d = np.load(npz_path, allow_pickle=True)
    roc_auc = d["roc_auc"]
    try:
        maybe = roc_auc.item()
        if isinstance(maybe, dict):
            if "macro" in maybe:
                val = float(maybe["macro"])
            else:
                val = float(np.mean([float(v) for v in maybe.values()]))
            d.close()
            return val
    except Exception:
        pass
    arr = np.array(roc_auc, dtype=float).ravel()
    d.close()
    return float(arr.mean())

print("Helper functions created!")

In [ ]:
# Extract AUC from all models
print("Extracting AUC scores...")

auc_results = {}
for model_name, file_path in files.items():
    try:
        if model_name == "MModal_Baseline":
            auc_val = auc_from_predictions(file_path)
        else:
            auc_val = auc_from_roc_npz(file_path)
        
        auc_results[model_name] = auc_val
        print(f"{model_name}: {auc_val:.4f}")
        
    except Exception as e:
        print(f"ERROR with {model_name}: {e}")

print(f"\nSuccessfully extracted AUC from {len(auc_results)}/6 models")

In [ ]:
# Install missing dependencies
import subprocess
import sys

def install_package(package):
    try:
        __import__(package)  # FIXED: was **import**
        print(f"✅ {package} already installed")
    except ImportError:
        print(f"📦 Installing {package}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])
        print(f"✅ {package} installed successfully")

install_package("openpyxl")

In [ ]:
import pandas as pd

excel_path = "model_statistics_all.xlsx"

# Load Excel robustly
tmp = pd.read_excel(excel_path, header=None)
header_row_idx = tmp.index[
    tmp.apply(lambda r: r.astype(str).str.contains(r"\bModel_ID\b", case=False, regex=True, na=False)).any(axis=1)
][0]

df = pd.read_excel(excel_path, header=header_row_idx)
df.columns = [str(c).strip() for c in df.columns]

print("Excel loaded successfully!")
print("Columns found:", list(df.columns))
print("Shape:", df.shape)

In [ ]:
# Extract macro F1 scores
macro = df[df["Model_ID"].astype(str).str.lower().eq("macro avg")][["Model_Name","F1-score","Support"]].copy()
macro.rename(columns={"Model_Name":"Model","F1-score":"F1"}, inplace=True)

# Updated name mapping with your preferred format
name_map = {
    "simple_CNN_multi": "MModal_Baseline",
    "augmentation_CNN_multi": "MM-augmentation", 
    "att_augm_CNN_multi": "MM-attention",
    "transformer_augm_CNN_multi": "MM-transformer",
    "transformer_attention_CNN_multi": "MM-transformer_attention",  # underscore for combined
    "ViT16_augm_CNN_multi": "MM-ViT16",
}

macro["Model"] = macro["Model"].replace(name_map)
macro["Support"] = pd.to_numeric(macro["Support"], errors="coerce").fillna(237)

print("F1 scores extracted:")
print(macro)

In [ ]:
# Update files dictionary to match F1 naming
files = {
    "MModal_Baseline": "model_v1_predictions.npz",
    "MM-augmentation": "roc_data_augmentation.npz", 
    "MM-attention": "roc_data_augmentation_attention.npz",
    "MM-transformer": "roc_data_transformer.npz",
    "MM-transformer_attention": "roc_data_combined.npz",
    "MM-ViT16": "roc_data_transformer_ViT-16.npz",
}

# Update colors dictionary to match
colors = {
    "MModal_Baseline": "#1f77b4",
    "MM-augmentation": "#ff7f0e", 
    "MM-attention": "#2ca02c",
    "MM-transformer": "#d62728",
    "MM-transformer_attention": "#8c564b",
    "MM-ViT16": "#9467bd",
}

print("Updated dictionaries to match F1 naming:")
for model in files.keys():
    print(f"{model}: {colors[model]}")

In [ ]:
# Re-extract AUC with updated naming
print("Re-extracting AUC with updated names...")

auc_results = {}
for model_name, file_path in files.items():
    try:
        if model_name == "MModal_Baseline":
            auc_val = auc_from_predictions(file_path)
        else:
            auc_val = auc_from_roc_npz(file_path)
        
        auc_results[model_name] = auc_val
        print(f"{model_name}: {auc_val:.4f}")
        
    except Exception as e:
        print(f"ERROR with {model_name}: {e}")

# Convert to DataFrame
auc_df = pd.DataFrame(list(auc_results.items()), columns=["Model", "AUC"])
print("\nAUC DataFrame:")
print(auc_df)

In [ ]:
# Merge AUC and F1 data
metrics = pd.merge(auc_df, macro[["Model","F1","Support"]], on="Model", how="inner")
metrics = metrics.sort_values("AUC", ascending=False)

print("Combined metrics table:")
print(metrics.round(4))

print(f"\nSuccessfully merged data for {len(metrics)} models")

In [ ]:
import matplotlib.pyplot as plt

# Add parameter counts
param_counts = {
    "MModal_Baseline": 165_923,
    "MM-augmentation": 165_923,
    "MM-attention": 15_273_075,
    "MM-transformer": 373_667,
    "MM-transformer_attention": 16_059_795,
    "MM-ViT16": 315_011,
}

metrics["Params"] = metrics["Model"].map(param_counts)

# Create bubble chart
plt.figure(figsize=(10, 7))

# Bubble sizes based on sqrt of parameters
max_params = metrics["Params"].max()
sizes = (metrics["Params"] ** 0.5) / (max_params ** 0.5) * 1200

for i, row in metrics.iterrows():
    plt.scatter(row["AUC"], row["F1"],
                s=sizes.iloc[i],
                color=colors[row["Model"]],
                alpha=0.7,
                edgecolor="black",
                linewidth=1,
                label=row["Model"])
    
    plt.text(row["AUC"] + 0.0005, row["F1"] + 0.002, 
             row["Model"], fontsize=10, weight="bold")

plt.xlabel("AUC (macro)", fontsize=13)
plt.ylabel("F1-score (macro)", fontsize=13)
plt.title("Model Performance: AUC vs F1\n(Bubble size = Model Parameters)", fontsize=14, fontweight='bold')
plt.grid(True, linestyle="--", alpha=0.6)

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.lines import Line2D

plt.figure(figsize=(10, 6))

# Calculate bubble sizes (using the parameter counts)
p = metrics["Params"].values
p_min, p_max = p.min(), p.max()
p_median = np.median(p)

# Log scaling for bubble sizes to match reference
sizes = ((np.log10(p) - np.log10(p_min)) / (np.log10(p_max) - np.log10(p_min))) * 1000 + 200

# Plot the bubbles
for i, row in metrics.iterrows():
    plt.scatter(row["AUC"], row["F1"], 
                s=sizes[i], 
                color=colors[row["Model"]], 
                alpha=0.6, 
                edgecolor='black', 
                linewidth=1)

# Set axis limits to match reference
plt.xlim(0.965, 0.995)
plt.ylim(0.84, 0.94)

plt.xlabel("AUC (macro)", fontsize=12)
plt.ylabel("F1-score (macro)", fontsize=12)
plt.grid(True, alpha=0.3)

# Create model legend (top right)
model_handles = []
for model in metrics["Model"]:
    handle = Line2D([0], [0], marker='o', color='w', 
                   markerfacecolor=colors[model], markersize=8, 
                   alpha=0.8, markeredgecolor='black')
    model_handles.append(handle)

legend1 = plt.legend(model_handles, metrics["Model"].tolist(), 
                    title="Models", loc='center left', 
                    bbox_to_anchor=(1.02, 0.7), fontsize=9)

# Create size legend (bottom right)
size_values = [p_min, p_median, p_max]
size_labels = [f"Small ({int(p_min):,})", f"Medium ({int(p_median):,})", f"Large ({int(p_max):,})"]
size_handles = []

for val, label in zip(size_values, size_labels):
    size = ((np.log10(val) - np.log10(p_min)) / (np.log10(p_max) - np.log10(p_min))) * 1000 + 200
    handle = Line2D([0], [0], marker='o', color='w', 
                   markerfacecolor='gray', markersize=np.sqrt(size/50), 
                   alpha=0.6, markeredgecolor='black')
    size_handles.append(handle)

legend2 = plt.legend(size_handles, size_labels, 
                    title="Bubble size = Parameters", 
                    loc='center left', bbox_to_anchor=(1.02, 0.3), fontsize=9)

plt.gca().add_artist(legend1)  # Keep both legends

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

# Add the missing color for MM-transformer_attention
pastel_colors = {
    "MM-attention":    "#B2DF8A",  # pastel green
    "MModal_Baseline": "#A6CEE3",  # pastel blue
    "MM-augmentation": "#FDBF6F",  # pastel orange
    "MM-transformer":  "#FB9A99",  # pastel red
    "MM-transformer_attention": "#D4B5D4",  # pastel lavender
    "MM-ViT16":        "#CAB2D6",  # pastel purple
}

# bubble sizes from params (log scaling for clearer separation)
p = metrics["Params"].astype(float).values
p_min, p_max = p.min(), p.max()
# continuous log10 scale mapped to [300, 1400] points^2
sizes = ( (np.log10(p) - np.log10(p_min)) / (np.log10(p_max) - np.log10(p_min) + 1e-9) ) * (1400-300) + 300
sizes = pd.Series(sizes, index=metrics.index)

plt.figure(figsize=(9,6))
# plot points (alpha for transparency), labels placed NEXT to each bubble
for i, r in metrics.iterrows():
    plt.scatter(r["AUC"], r["F1"],
                s=sizes.loc[i],
                color=pastel_colors[r["Model"]],
                alpha=0.55)
    # small offset to the right/up so text sits next to the bubble
    plt.text(r["AUC"] + 0.0008, r["F1"] + 0.0008, r["Model"],
             ha="left", va="bottom", fontsize=10, color="black")

# axis ranges
plt.xlim(0.965, 0.997)
plt.ylim(0.84, 0.94)
plt.xlabel("AUC (macro)", fontsize=13)
plt.ylabel("F1-score (macro)", fontsize=13)
plt.grid(True, linestyle="--", alpha=0.6)

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

# Add the missing color for MM-transformer_attention
pastel_colors = {
    "MM-attention":    "#B2DF8A",  # pastel green
    "MModal_Baseline": "#A6CEE3",  # pastel blue
    "MM-augmentation": "#FDBF6F",  # pastel orange
    "MM-transformer":  "#FB9A99",  # pastel red
    "MM-transformer_attention": "#D4B5D4",  # pastel lavender
    "MM-ViT16":        "#CAB2D6",  # pastel purple
}

# bubble sizes from params (log scaling for clearer separation)
p = metrics["Params"].astype(float).values
p_min, p_max = p.min(), p.max()
sizes = ( (np.log10(p) - np.log10(p_min)) / (np.log10(p_max) - np.log10(p_min) + 1e-9) ) * (1400-300) + 300
sizes = pd.Series(sizes, index=metrics.index)

plt.figure(figsize=(9,6))

# Custom label offsets to prevent overlap
label_offsets = {
    "MM-transformer_attention": (0.0005, -0.005),  # Move down to avoid overlap
    "MM-augmentation": (0.0008, 0.0008),          # Default position
}

# plot points and labels
for i, r in metrics.iterrows():
    plt.scatter(r["AUC"], r["F1"],
                s=sizes.loc[i],
                color=pastel_colors[r["Model"]],
                alpha=0.55)
    
    # Use custom offset if available, otherwise default
    if r["Model"] in label_offsets:
        offset_x, offset_y = label_offsets[r["Model"]]
    else:
        offset_x, offset_y = 0.0008, 0.0008
    
    plt.text(r["AUC"] + offset_x, r["F1"] + offset_y, r["Model"],
             ha="left", va="bottom", fontsize=10, color="black")

plt.xlim(0.97, 0.994)
plt.ylim(0.86, 0.95)
plt.xlabel("AUC (macro)", fontsize=13)
plt.ylabel("F1-score (macro)", fontsize=13)
plt.grid(True, linestyle="--", alpha=0.6)

# Add color legend (models)
color_handles = [
    Line2D([0],[0], marker='o', linestyle='', markersize=10,
           markerfacecolor=pastel_colors[m], markeredgecolor='black',
           alpha=0.55, label=m)
    for m in metrics["Model"]
]
leg1 = plt.legend(handles=color_handles, title="Models",
                  loc="upper left", bbox_to_anchor=(1.10, 1.0), frameon=True)
leg1.get_frame().set_edgecolor("black")
leg1.get_frame().set_linewidth(1.0)
plt.gca().add_artist(leg1)

# Add size legend
ticks_params = np.array([p_min, np.median(p), p_max], dtype=float)
ticks_sizes = ( (np.log10(ticks_params) - np.log10(p_min)) /
                (np.log10(p_max) - np.log10(p_min) + 1e-9) ) * (1400-300) + 300
size_labels = [
    f"Small  ({int(ticks_params[0]):,})",
    f"Medium ({int(ticks_params[1]):,})",
    f"Large  ({int(ticks_params[2]):,})",
]
size_handles = [
    plt.scatter([], [], s=ticks_sizes[j], facecolor="gray", alpha=0.35,
                edgecolor="black", linewidth=0.9, label=size_labels[j])
    for j in range(3)
]
leg2 = plt.legend(handles=size_handles, title="Bubble size = Parameters",
                  loc="upper left", bbox_to_anchor=(1.10, 0.55), frameon=True)
leg2.get_frame().set_edgecolor("black")
leg2.get_frame().set_linewidth(1.5)

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_curve, auc

def macro_from_npz(npz_path):
    """Extract macro ROC from NPZ files that have fpr/tpr/roc_auc"""
    d = np.load(npz_path, allow_pickle=True)
    fpr = d["fpr"].item() if hasattr(d["fpr"], "item") else d["fpr"]
    tpr = d["tpr"].item() if hasattr(d["tpr"], "item") else d["tpr"]
    aucd = d["roc_auc"].item() if hasattr(d["roc_auc"], "item") else d["roc_auc"]
    return fpr["macro"], tpr["macro"], aucd["macro"]

def macro_from_scores(npz_path):
    """Extract macro ROC from baseline model (y_test, y_score)"""
    d = np.load(npz_path, allow_pickle=True)
    y_true = d["y_test"]
    y_prob = d["y_score"]
    classes = np.arange(y_prob.shape[1])
    Y = label_binarize(y_true, classes=classes)
    
    fpr, tpr, roc_auc = {}, {}, {}
    for i in classes:
        fpr[i], tpr[i], _ = roc_curve(Y[:, i], y_prob[:, i])
        roc_auc[i] = auc(fpr[i], tpr[i])
    
    # Macro average
    all_fpr = np.unique(np.concatenate([fpr[i] for i in classes]))
    mean_tpr = np.zeros_like(all_fpr)
    for i in classes:
        mean_tpr += np.interp(all_fpr, fpr[i], tpr[i])
    mean_tpr /= len(classes)
    
    return all_fpr, mean_tpr, auc(all_fpr, mean_tpr)

print("ROC helper functions created!")

In [ ]:
plt.figure(figsize=(8, 6))

# Plot ROC curve for each model
for name, path in files.items():
    if name == "MModal_Baseline":
        fpr, tpr, auc_macro = macro_from_scores(path)
    else:
        fpr, tpr, auc_macro = macro_from_npz(path)
    
    plt.plot(fpr, tpr, 
             linewidth=2.5, 
             color=pastel_colors[name],
             label=f"{name} (AUC={auc_macro:.3f})",
             alpha=0.9)

# Add diagonal reference line
plt.plot([0, 1], [0, 1], 
         color="gray", 
         linewidth=1, 
         linestyle="--", 
         alpha=0.7,
         label="Random Classifier")

plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel("False Positive Rate", fontsize=12)
plt.ylabel("True Positive Rate", fontsize=12)
plt.title("ROC Curves - Macro Average", fontsize=14, fontweight='bold')
plt.legend(loc="lower right", fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Updated pastel colors - changed MM-transformer_attention to yellow
pastel_colors = {
    "MM-attention":    "#B2DF8A",  # pastel green
    "MModal_Baseline": "#A6CEE3",  # pastel blue
    "MM-augmentation": "#FDBF6F",  # pastel orange
    "MM-transformer":  "#FB9A99",  # pastel red
    "MM-transformer_attention": "#FFFFB3",  # pastel yellow (changed from purple)
    "MM-ViT16":        "#CAB2D6",  # pastel purple
}

In [ ]:
import pandas as pd

# Load validation results from Excel
excel_path = "model_statistics_all.xlsx"
tmp = pd.read_excel(excel_path, header=None)
header_row_idx = tmp.index[
    tmp.apply(lambda r: r.astype(str).str.contains(r"\bModel_ID\b", case=False, regex=True, na=False)).any(axis=1)
][0]
df = pd.read_excel(excel_path, header=header_row_idx)
df.columns = [str(c).strip() for c in df.columns]

# Extract validation accuracy data
acc_col = "Mean val accuracy (post-training 5fold)"
acc_std_col = "Standard Deviation"

summary = df.dropna(subset=[acc_col]).copy()
summary["Model"] = summary["Model_Name"].replace(name_map)
summary = summary[summary["Model"].isin(files.keys())]

# Convert to numeric
val_acc_means = pd.to_numeric(summary[acc_col], errors="coerce")
val_acc_stds = pd.to_numeric(summary[acc_std_col], errors="coerce")

print("Validation accuracy data extracted!")
print("Models found:", summary["Model"].tolist())

In [ ]:
# Clean up the data - remove duplicates and ensure we have all models
summary_clean = summary.drop_duplicates(subset=['Model']).copy()

# Check if we have all expected models
expected_models = list(files.keys())
found_models = summary_clean["Model"].tolist()

print("Expected models:", expected_models)
print("Found models:", found_models)
print("Missing models:", [m for m in expected_models if m not in found_models])

# Filter to only our 6 models and sort consistently
model_order = ["MModal_Baseline", "MM-augmentation", "MM-attention", "MM-transformer", "MM-transformer_attention", "MM-ViT16"]
summary_final = summary_clean[summary_clean["Model"].isin(model_order)].copy()
summary_final = summary_final.set_index("Model").reindex(model_order).reset_index()

print("\nFinal validation data:")
print(summary_final[["Model", acc_col, acc_std_col]])

In [ ]:
import matplotlib.pyplot as plt

# Extract the data
models = summary_final["Model"].tolist()
val_means = summary_final[acc_col].values
val_stds = summary_final[acc_std_col].values

# Get colors in the same order
model_colors = [pastel_colors[model] for model in models]

plt.figure(figsize=(10, 6))

# Create bar chart with error bars
bars = plt.bar(models, val_means, 
               yerr=val_stds, 
               capsize=5,
               color=model_colors, 
               alpha=0.8, 
               edgecolor='black', 
               linewidth=0.5)

# Add value labels on top of error bars (not just bars)
for bar, mean_val, std_val in zip(bars, val_means, val_stds):
    plt.text(bar.get_x() + bar.get_width()/2, 
             bar.get_height() + std_val + 0.005,  # Changed: add std_val + small margin
             f'{mean_val:.3f}', 
             ha='center', va='bottom', 
             fontsize=10, fontweight='bold')

plt.ylabel("Validation Accuracy (5-fold CV)", fontsize=12)
plt.xlabel("Models", fontsize=12)
plt.title("Mean Validation Accuracy ± Standard Deviation", fontsize=14, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', alpha=0.3)
plt.ylim(0.85, 0.95)

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np

def create_radar_chart(class_name):
    """Create radar chart for a specific class"""
    # Filter data for this class
    class_data = df[df["Model_ID"].astype(str).str.lower().eq(class_name.lower())].copy()
    
    if class_data.empty:
        print(f"No data found for class '{class_name}'")
        return
    
    # Map model names
    class_data["Model"] = class_data["Model_Name"].replace(name_map)
    class_data = class_data[class_data["Model"].isin(files.keys())]
    
    # Set up for radar chart
    metrics = ["Precision", "Recall", "F1-score"]
    angles = np.linspace(0, 2*np.pi, len(metrics), endpoint=False)
    angles = np.concatenate([angles, angles[:1]])  # Close the circle
    
    print(f"Creating radar chart for {class_name}...")
    print("Available data:")
    print(class_data[["Model"] + metrics])

# Test with one class first
create_radar_chart("Response")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def radar_plot_for_class(class_name):
    # Filter rows for this class
    sub = df[df["Model_ID"].astype(str).str.strip().str.lower().eq(class_name.lower())].copy()
    if sub.empty:
        print(f"No rows found for class '{class_name}'")
        return
    
    # Map model names and filter to our models
    sub["Model"] = sub["Model_Name"].replace(name_map)
    sub = sub[sub["Model"].isin(files.keys())]
    sub = sub.set_index("Model").dropna(subset=["Precision","Recall","F1-score"])
    
    metrics = ["Precision","Recall","F1-score"]
    angles = np.linspace(0, 2*np.pi, len(metrics), endpoint=False)
    angles = np.concatenate([angles, angles[:1]])  # close the loop
    
    fig, ax = plt.subplots(figsize=(6, 6), subplot_kw=dict(polar=True))
    ax.set_title(f"{class_name} Class - Performance Metrics", fontsize=14, fontweight="bold", pad=20)
    
    # Set up the radar cha# Create 1x3 horizontal layout
fig, axes = plt.subplots(1, 3, figsize=(15, 5), subplot_kw=dict(polar=True))
classes = ["Response", "Stable", "Non-Response"]

for ax, class_name in zip(axes, classes):
    sub = df[df["Model_ID"].astype(str).str.strip().str.lower().eq(class_name.lower())].copy()
    sub["Model"] = sub["Model_Name"].replace(name_map)
    sub = sub[sub["Model"].isin(files.keys())]
    sub = sub.set_index("Model").dropna(subset=["Precision","Recall","F1-score"])
    
    metrics = ["Precision","Recall","F1-score"]
    angles = np.linspace(0, 2*np.pi, len(metrics), endpoint=False)
    angles = np.concatenate([angles, angles[:1]])
    
    ax.set_title(class_name, fontsize=13, fontweight="bold")
    ax.set_ylim(0.0, 1.0)
    ax.set_yticks([0.6, 0.8, 1.0])
    ax.set_xticks(angles[:-1])
    
    # Custom labels with shifted Precision
    ax.set_xticklabels(["", "Recall", "F1-score"], fontsize=10)
    ax.text(angles[0] + 0.1, 1.05, "Precision", ha="center", va="center", fontsize=10)  # Shifted right
    
    for model in sub.index:
        if model in radar_colors:
            values = sub.loc[model, metrics].values.astype(float)
            values = np.concatenate([values, values[:1]])
            ax.plot(angles, values, color=radar_colors[model], 
                   linewidth=2.5, label=model, alpha=0.7)  # Changed from 0.9 to 0.7
            ax.fill(angles, values, color=radar_colors[model], alpha=0.1)  # Changed from 0.15 to 0.1

# Single legend for all subplots, positioned far right
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="center left", bbox_to_anchor=(1.0, 0.5), fontsize=10)

plt.tight_layout()
plt.show()rt
    ax.set_ylim(0.0, 1.0)
    ax.set_yticks([0.6, 0.7, 0.8, 0.9, 1.0])
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(metrics, fontsize=11)
    
    # Plot each model
    for model in sub.index:
        if model in pastel_colors:
            values = sub.loc[model, metrics].values.astype(float)
            values = np.concatenate([values, values[:1]])
            ax.plot(angles, values, color=pastel_colors[model], 
                   linewidth=2.5, label=model, alpha=0.8)
            ax.fill(angles, values, color=pastel_colors[model], alpha=0.2)
    
    ax.legend(loc="upper right", bbox_to_anchor=(1.3, 1.0))
    plt.tight_layout()
    plt.show()

# Create radar charts for all three classes
for class_name in ["Response", "Stable", "Non-Response"]:
    radar_plot_for_class(class_name)

In [ ]:
# Create 1x3 horizontal layout
fig, axes = plt.subplots(1, 3, figsize=(15, 5), subplot_kw=dict(polar=True))
classes = ["Response", "Stable", "Non-Response"]

for ax, class_name in zip(axes, classes):
    sub = df[df["Model_ID"].astype(str).str.strip().str.lower().eq(class_name.lower())].copy()
    sub["Model"] = sub["Model_Name"].replace(name_map)
    sub = sub[sub["Model"].isin(files.keys())]
    sub = sub.set_index("Model").dropna(subset=["Precision","Recall","F1-score"])
    
    metrics = ["Precision","Recall","F1-score"]
    angles = np.linspace(0, 2*np.pi, len(metrics), endpoint=False)
    angles = np.concatenate([angles, angles[:1]])
    
    ax.set_title(class_name, fontsize=13, fontweight="bold")
    ax.set_ylim(0.0, 1.0)
    ax.set_yticks([0.6, 0.8, 1.0])
    ax.set_xticks(angles[:-1])
    
    # Custom labels with shifted Precision
    ax.set_xticklabels(["", "Recall", "F1-score"], fontsize=10)
    ax.text(angles[0] + 0.1, 1.05, "Precision", ha="center", va="center", fontsize=10)  # Shifted right
    
    for model in sub.index:
        if model in radar_colors:
            values = sub.loc[model, metrics].values.astype(float)
            values = np.concatenate([values, values[:1]])
            ax.plot(angles, values, color=radar_colors[model], 
                   linewidth=2.5, label=model, alpha=0.7)  # Changed from 0.9 to 0.7
            ax.fill(angles, values, color=radar_colors[model], alpha=0.1)  # Changed from 0.15 to 0.1

# Single legend for all subplots, positioned far right
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="center left", bbox_to_anchor=(1.0, 0.5), fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
# Extract test performance data from Excel
test_loss_col = "Test Loss"
test_acc_col = "Test Accuracy"

# Get one row per model (avoid duplicates)
test_data = df.dropna(subset=[test_loss_col, test_acc_col]).copy()
test_data["Model"] = test_data["Model_Name"].replace(name_map)
test_data = test_data[test_data["Model"].isin(files.keys())]
test_data = test_data.drop_duplicates(subset=['Model'])

# Convert to numeric
test_losses = pd.to_numeric(test_data[test_loss_col], errors="coerce")
test_accuracies = pd.to_numeric(test_data[test_acc_col], errors="coerce")

print("Test performance data:")
for i, row in test_data.iterrows():
    model = row["Model"]
    loss = test_losses.iloc[list(test_data.index).index(i)]
    acc = test_accuracies.iloc[list(test_data.index).index(i)]
    print(f"{model}: Loss={loss:.4f}, Accuracy={acc:.4f}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Ensure proper model order and extract values
model_order = ["MModal_Baseline", "MM-augmentation", "MM-attention", "MM-transformer", "MM-transformer_attention", "MM-ViT16"]
test_data_ordered = test_data.set_index("Model").reindex(model_order).reset_index()

models = test_data_ordered["Model"].tolist()
test_losses = test_data_ordered[test_loss_col].values
test_accs = test_data_ordered[test_acc_col].values

# Colors for loss and accuracy
loss_color = "#ff9d9a"  # pastel red
acc_color = "#8de5a1"   # pastel green

x = np.arange(len(models))
width = 0.35

plt.figure(figsize=(10, 6))

# Create grouped bars
bars1 = plt.bar(x - width/2, test_losses, width, 
                color=loss_color, alpha=0.8, label="Test Loss")
bars2 = plt.bar(x + width/2, test_accs, width, 
                color=acc_color, alpha=0.8, label="Test Accuracy")

# Add value labels
for bar in bars1:
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)

for bar in bars2:
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)

plt.xlabel("Models", fontsize=12)
plt.ylabel("Value", fontsize=12)
plt.title("Test Loss & Test Accuracy per Model", fontsize=14, fontweight='bold')
plt.xticks(x, models, rotation=45, ha='right')
plt.legend(fontsize=11)
plt.grid(axis='y', alpha=0.3)
plt.ylim(0, 1.1)

plt.tight_layout()
plt.show()

In [ ]:
# Extract validation loss data
val_loss_col = "Mean val loss (post-training 5fold)"
val_loss_std_col = "Standard Deviation.1"  # The second Standard Deviation column

# Get validation loss data (same as we did for accuracy)
val_loss_data = df.dropna(subset=[val_loss_col]).copy()
val_loss_data["Model"] = val_loss_data["Model_Name"].replace(name_map)
val_loss_data = val_loss_data[val_loss_data["Model"].isin(files.keys())]
val_loss_data = val_loss_data.drop_duplicates(subset=['Model'])

# Order consistently
val_loss_ordered = val_loss_data.set_index("Model").reindex(model_order).reset_index()

models = val_loss_ordered["Model"].tolist()
loss_means = pd.to_numeric(val_loss_ordered[val_loss_col], errors="coerce").values
loss_stds = pd.to_numeric(val_loss_ordered[val_loss_std_col], errors="coerce").values

# Get colors for each model
model_colors = [pastel_colors[model] for model in models]

plt.figure(figsize=(10, 6))

# Create bar chart with error bars
bars = plt.bar(models, loss_means, 
               yerr=loss_stds, 
               capsize=5,
               color=model_colors, 
               alpha=0.8, 
               edgecolor='black', 
               linewidth=0.5)

# Add value labels above error bars
for bar, mean_val, std_val in zip(bars, loss_means, loss_stds):
    plt.text(bar.get_x() + bar.get_width()/2, 
             bar.get_height() + std_val + 0.01,
             f'{mean_val:.3f}', 
             ha='center', va='bottom', 
             fontsize=10, fontweight='bold')

plt.ylabel("Validation Loss (5-fold CV)", fontsize=12)
plt.xlabel("Models", fontsize=12)
plt.title("Mean Validation Loss ± Standard Deviation", fontsize=14, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Create parameter efficiency analysis
plt.figure(figsize=(12, 5))

# Subplot 1: AUC vs Parameters
plt.subplot(1, 2, 1)
model_colors_ordered = [pastel_colors[model] for model in metrics["Model"]]

plt.scatter(metrics["Params"], metrics["AUC"], 
           c=model_colors_ordered, s=100, alpha=0.7, edgecolor='black')

for i, row in metrics.iterrows():
    plt.annotate(row["Model"], 
                (row["Params"], row["AUC"]), 
                xytext=(5, 5), textcoords='offset points', fontsize=9)

plt.xlabel("Parameter Count", fontsize=12)
plt.ylabel("AUC Score", fontsize=12)
plt.title("Model Complexity vs AUC Performance", fontsize=13, fontweight='bold')
plt.xscale('log')  # Log scale for parameters
plt.grid(True, alpha=0.3)

# Subplot 2: F1 vs Parameters  
plt.subplot(1, 2, 2)
plt.scatter(metrics["Params"], metrics["F1"], 
           c=model_colors_ordered, s=100, alpha=0.7, edgecolor='black')

for i, row in metrics.iterrows():
    plt.annotate(row["Model"], 
                (row["Params"], row["F1"]), 
                xytext=(5, 5), textcoords='offset points', fontsize=9)

plt.xlabel("Parameter Count", fontsize=12)
plt.ylabel("F1 Score", fontsize=12)
plt.title("Model Complexity vs F1 Performance", fontsize=13, fontweight='bold')
plt.xscale('log')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Recreate the metrics DataFrame from our earlier work
metrics_data = []
for model_name, file_path in files.items():
    if model_name == "MModal_Baseline":
        auc_val = auc_from_predictions(file_path)
    else:
        auc_val = auc_from_roc_npz(file_path)
    metrics_data.append({"Model": model_name, "AUC": auc_val})

# Convert to DataFrame
auc_df = pd.DataFrame(metrics_data)

# Merge with F1 data (from the macro DataFrame we created earlier)
metrics = pd.merge(auc_df, macro[["Model","F1","Support"]], on="Model", how="inner")
metrics = metrics.sort_values("AUC", ascending=False)

# Add parameter counts
metrics["Params"] = metrics["Model"].map(param_counts)

print("Recreated metrics DataFrame:")
print(metrics)

In [ ]:
# Redefine colors as a proper dictionary
model_colors_dict = {
    "MModal_Baseline": "#1f77b4",  # blue
    "MM-augmentation": "#ff7f0e",  # orange
    "MM-attention": "#2ca02c",     # green
    "MM-transformer": "#d62728",   # red
    "MM-transformer_attention": "#9467bd",  # purple
    "MM-ViT16": "#8c564b",         # brown
}

# Now create the plot
plt.figure(figsize=(12, 5))

# Subplot 1: AUC vs Parameters
plt.subplot(1, 2, 1)
model_colors_ordered = [model_colors_dict[model] for model in metrics["Model"]]

plt.scatter(metrics["Params"], metrics["AUC"], 
           c=model_colors_ordered, s=100, alpha=0.7, edgecolor='black')

for i, row in metrics.iterrows():
    plt.annotate(row["Model"], 
                (row["Params"], row["AUC"]), 
                xytext=(5, 5), textcoords='offset points', fontsize=9)

plt.xlabel("Parameter Count", fontsize=12)
plt.ylabel("AUC Score", fontsize=12)
plt.title("Model Complexity vs AUC Performance", fontsize=13, fontweight='bold')
plt.xscale('log')
plt.grid(True, alpha=0.3)

# Subplot 2: F1 vs Parameters  
plt.subplot(1, 2, 2)
plt.scatter(metrics["Params"], metrics["F1"], 
           c=model_colors_ordered, s=100, alpha=0.7, edgecolor='black')

for i, row in metrics.iterrows():
    plt.annotate(row["Model"], 
                (row["Params"], row["F1"]), 
                xytext=(5, 5), textcoords='offset points', fontsize=9)

plt.xlabel("Parameter Count", fontsize=12)
plt.ylabel("F1 Score", fontsize=12)
plt.title("Model Complexity vs F1 Performance", fontsize=13, fontweight='bold')
plt.xscale('log')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Use the pastel colors you specified
model_colors_dict = {
    "MM-attention":    "#B2DF8A",  # pastel green
    "MModal_Baseline": "#A6CEE3",  # pastel blue
    "MM-augmentation": "#FDBF6F",  # pastel orange
    "MM-transformer":  "#FB9A99",  # pastel red
    "MM-transformer_attention": "#E5C494",  # pastel yellow
    "MM-ViT16":        "#CAB2D6",  # pastel purple
}

plt.figure(figsize=(12, 5))

# Subplot 1: AUC vs Parameters
plt.subplot(1, 2, 1)
model_colors_ordered = [model_colors_dict[model] for model in metrics["Model"]]

plt.scatter(metrics["Params"], metrics["AUC"], 
           c=model_colors_ordered, s=100, alpha=0.7, edgecolor='black')

# Fixed label positioning to avoid overlap
for i, row in metrics.iterrows():
    plt.annotate(row["Model"], 
                (row["Params"], row["AUC"]), 
                xytext=(10, 10), textcoords='offset points', fontsize=9,
                bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.7))

plt.xlabel("Parameter Count", fontsize=12)
plt.ylabel("AUC Score", fontsize=12)
plt.title("Model Complexity vs AUC Performance", fontsize=13, fontweight='bold')
plt.xscale('log')
plt.xlim(100000, 20000000)  # Increased limits
plt.ylim(0.96, 1.0)         # Increased limits
plt.grid(True, alpha=0.3)

# Subplot 2: F1 vs Parameters  
plt.subplot(1, 2, 2)
plt.scatter(metrics["Params"], metrics["F1"], 
           c=model_colors_ordered, s=100, alpha=0.7, edgecolor='black')

# Fixed label positioning to avoid overlap
for i, row in metrics.iterrows():
    plt.annotate(row["Model"], 
                (row["Params"], row["F1"]), 
                xytext=(10, 10), textcoords='offset points', fontsize=9,
                bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.7))

plt.xlabel("Parameter Count", fontsize=12)
plt.ylabel("F1 Score", fontsize=12)
plt.title("Model Complexity vs F1 Performance", fontsize=13, fontweight='bold')
plt.xscale('log')
plt.xlim(100000, 20000000)  # Increased limits
plt.ylim(0.85, 0.95)        # Increased limits
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Create a Pareto frontier showing the trade-off between performance and efficiency
plt.figure(figsize=(10, 6))

# Calculate efficiency metric (AUC per million parameters)
metrics["Efficiency"] = metrics["AUC"] / (metrics["Params"] / 1_000_000)

# Plot performance vs efficiency
for i, row in metrics.iterrows():
    plt.scatter(row["Efficiency"], row["AUC"], 
               s=200, color=model_colors_dict[row["Model"]], 
               alpha=0.7, edgecolor='black', linewidth=2)
    plt.annotate(row["Model"], (row["Efficiency"], row["AUC"]), 
                xytext=(10, 5), textcoords='offset points', fontsize=10)

plt.xlabel("Efficiency (AUC per Million Parameters)", fontsize=12)
plt.ylabel("AUC Score", fontsize=12)
plt.title("Performance-Efficiency Trade-off Analysis", fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Create a comprehensive performance heatmap
import seaborn as sns

# Prepare data for heatmap (normalize all metrics to 0-1)
heatmap_data = metrics.copy()
cols_to_normalize = ['AUC', 'F1', 'Params']

# Invert parameters (lower is better for complexity)
heatmap_data['Simplicity'] = 1 / heatmap_data['Params'] * 1e6

# Normalize all metrics to 0-1 scale
for col in ['AUC', 'F1', 'Simplicity']:
    heatmap_data[col] = (heatmap_data[col] - heatmap_data[col].min()) / (heatmap_data[col].max() - heatmap_data[col].min())

# Create heatmap matrix
heatmap_matrix = heatmap_data[['AUC', 'F1', 'Simplicity']].T
heatmap_matrix.columns = heatmap_data['Model']

plt.figure(figsize=(12, 6))
sns.heatmap(heatmap_matrix, annot=True, fmt='.3f', cmap='RdYlGn', 
            cbar_kws={'label': 'Normalized Score (0-1)'})
plt.title('Multi-Dimensional Model Performance Matrix\n(Higher is Better)', fontsize=14, fontweight='bold')
plt.ylabel('Performance Metrics', fontsize=12)
plt.xlabel('Models', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Install seaborn
import subprocess
import sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "seaborn"])

import seaborn as sns
print("Seaborn installed successfully!")

In [ ]:
# Create a hierarchical view of model complexity
fig, ax = plt.subplots(figsize=(12, 8))

# Group models by architectural approach
architecture_groups = {
    'Basic CNN': ['MModal_Baseline', 'MM-augmentation'],
    'Attention-Based': ['MM-attention', 'MM-transformer_attention'], 
    'Transformer': ['MM-transformer', 'MM-ViT16']
}

y_pos = 0
for group, models in architecture_groups.items():
    # Draw group label
    ax.text(-0.1, y_pos + len(models)/2 - 0.5, group, 
            fontsize=12, fontweight='bold', ha='right', va='center')
    
    for i, model in enumerate(models):
        model_data = metrics[metrics['Model'] == model].iloc[0]
        
        # Draw model bar (AUC)
        bar_width = model_data['AUC'] * 10
        ax.barh(y_pos + i, bar_width, height=0.6, 
                color=model_colors_dict[model], alpha=0.7, edgecolor='black')
        
        # Add model name and metrics
        ax.text(bar_width + 0.1, y_pos + i, 
                f"{model}\nAUC: {model_data['AUC']:.3f}, F1: {model_data['F1']:.3f}\nParams: {model_data['Params']:,}",
                fontsize=9, va='center')
    
    y_pos += len(models) + 1

ax.set_xlabel('AUC Score (×10 for visualization)', fontsize=12)
ax.set_title('Hierarchical Model Architecture Performance', fontsize=14, fontweight='bold')
ax.set_yticks([])
ax.grid(True, axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Extract confidence data for each model
confidence_data = {
    'MModal_Baseline': {'accuracy': 0.8945, 'avg_conf': 0.7909, 'correct_conf': 0.8127, 'incorrect_conf': 0.6059},
    'MM-augmentation': {'accuracy': 0.9156, 'avg_conf': 0.7116, 'correct_conf': 0.7258, 'incorrect_conf': 0.5582},
    'MM-attention': {'accuracy': 0.9367, 'avg_conf': 0.9162, 'correct_conf': 0.9276, 'incorrect_conf': 0.7469},
    'MM-transformer': {'accuracy': 0.9198, 'avg_conf': 0.9306, 'correct_conf': 0.9445, 'incorrect_conf': 0.7721},
    'MM-ViT16': {'accuracy': 0.9536, 'avg_conf': 0.9202, 'correct_conf': 0.9268, 'incorrect_conf': 0.7852},
    'MM-transformer_attention': {'accuracy': 0.8736, 'avg_conf': 0.8780, 'correct_conf': 0.8780, 'incorrect_conf': 0.6330}
}

# Create confidence analysis plot
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Plot 1: Confidence Calibration
models = list(confidence_data.keys())
correct_confs = [confidence_data[m]['correct_conf'] for m in models]
incorrect_confs = [confidence_data[m]['incorrect_conf'] for m in models]
colors_list = [model_colors_dict[m] for m in models]

x = range(len(models))
width = 0.35

axes[0].bar([i - width/2 for i in x], correct_confs, width, 
           label='Correct Predictions', alpha=0.8, color='green')
axes[0].bar([i + width/2 for i in x], incorrect_confs, width, 
           label='Incorrect Predictions', alpha=0.8, color='red')

axes[0].set_xlabel('Models')
axes[0].set_ylabel('Average Confidence')
axes[0].set_title('Model Calibration: Confidence by Prediction Correctness')
axes[0].set_xticks(x)
axes[0].set_xticklabels(models, rotation=45, ha='right')
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)

# Plot 2: Confidence vs Accuracy Scatter
accuracies = [confidence_data[m]['accuracy'] for m in models]
avg_confs = [confidence_data[m]['avg_conf'] for m in models]

axes[1].scatter(avg_confs, accuracies, c=colors_list, s=150, alpha=0.7, edgecolor='black')
for i, model in enumerate(models):
    axes[1].annotate(model, (avg_confs[i], accuracies[i]), 
                    xytext=(5, 5), textcoords='offset points', fontsize=9)

axes[1].plot([0.5, 1], [0.5, 1], 'k--', alpha=0.5, label='Perfect Calibration')
axes[1].set_xlabel('Average Confidence')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Confidence-Accuracy Relationship')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()